# 01 - Synthetic CNC Data Creation

**Goal:** Generate a realistic CNC machining dataset (2,000 parts) using machining domain knowledge.

**Why synthetic?** Real manufacturing data is confidential. The data is generated using cause-and-effect rules:
- Higher tool wear, speed and feed → higher vibration and cutting temperature
- Higher coolant flow → lower temperature
- Titanium → highest heat and defect risk
- Defect probability is derived from vibration, temperature, tool wear, coolant flow and material

**Real-world messiness added on purpose:** ~3% missing sensor values and 20 duplicate rows.

**Output:** `data/raw/cnc_machining_data.csv`

**Limitation:** Because the data is simulated, patterns are cleaner than in a real factory.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
np.random.seed(42)
n = 2000   # 2000 parts

# ---------- 1. Independent features ----------
material = np.random.choice(["Steel", "Aluminium", "Titanium"],
                            size=n, p=[0.5, 0.3, 0.2])

spindle_speed = np.random.normal(1500, 300, n).clip(600, 2500)
feed_rate     = np.random.normal(200, 40, n).clip(80, 350)
tool_wear     = np.random.uniform(0, 240, n)          # minutes
coolant_flow  = np.random.normal(8, 1.5, n).clip(2, 14)

# ---------- 2. Dependent features (physics logic) ----------
material_heat = pd.Series(material).map(
    {"Aluminium": 0, "Steel": 15, "Titanium": 35}
).values

cutting_temp = (120 + 0.03 * spindle_speed + 0.1 * feed_rate
                + 0.15 * tool_wear - 4 * coolant_flow
                + material_heat + np.random.normal(0, 8, n))

vibration = (0.5 + 0.0008 * spindle_speed + 0.002 * feed_rate
             + 0.008 * tool_wear + np.random.normal(0, 0.3, n)).clip(0.1)

# ---------- 3. Defect probability ----------
def zscore(x):
    return (x - x.mean()) / x.std()

material_risk = pd.Series(material).map(
    {"Aluminium": -0.3, "Steel": 0.0, "Titanium": 0.5}
).values

risk_score = (1.2 * zscore(vibration)
              + 1.0 * zscore(cutting_temp)
              + 0.8 * zscore(tool_wear)
              - 0.5 * zscore(coolant_flow)
              + material_risk)

logit = -2.2 + risk_score
defect_prob = 1 / (1 + np.exp(-logit))        # sigmoid: score → probability (0 to 1)
defect = np.random.binomial(1, defect_prob)   # coin flip with that probability

# ---------- 4. DataFrame ----------
df = pd.DataFrame({
    "part_id": range(1, n + 1),
    "material_type": material,
    "spindle_speed_rpm": spindle_speed.round(0),
    "feed_rate_mm_min": feed_rate.round(1),
    "cutting_temp_c": cutting_temp.round(1),
    "vibration_mm_s": vibration.round(2),
    "tool_wear_min": tool_wear.round(1),
    "coolant_flow_lpm": coolant_flow.round(2),
    "defect": defect
})


In [ ]:
# 3% missing values in vibration and coolant (sensor failure)
df.loc[df.sample(frac=0.03, random_state=1).index, "vibration_mm_s"] = np.nan
df.loc[df.sample(frac=0.03, random_state=2).index, "coolant_flow_lpm"] = np.nan

# 20 duplicate rows (data entry mistake)
df = pd.concat([df, df.sample(20, random_state=3)], ignore_index=True)

print("Shape:", df.shape)

In [ ]:
print(df["defect"].value_counts())
print(df["defect"].value_counts(normalize=True) * 100)

In [ ]:
df.to_csv("../data/raw/cnc_machining_data.csv", index=False)
print("Saved ✅")